# M1 Lab 2 — First Eval (LLM-as-a-Judge)

Runs Version A (Concise) vs Version B (Narrative) on the Q4 Marketing email, then has a judge model score both against the golden-set criteria in `eval-harness-proof.md`.

**Setup:** put `GOOGLE_API_KEY=...` in a `.env` file in this folder (already gitignored). Deps are installed in the project `.venv` (`google-genai`, `python-dotenv`).

In [4]:
pip install --upgrade "openai>=1.0"

^C
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import json
import statistics
from typing import Dict, Any, List
from openai import OpenAI

# Initialize the Openai client to read your API Key from the environment variable
client = OpenAI(
    # This is the default path. You can configure it based on the service location
    base_url="https://ark.ap-southeast.bytepluses.com/api/v3",
    # Get your API Key from the environment variable
    api_key=""
)

GENERATOR_MODEL = "deepseek-v4-1-flash-260910"
JUDGE_MODEL = "seed-2-0-mini-260428"

PROMPT_A = """You are Ascend IQ, an executive data assistant for C-suite leaders. 
Answer queries directly in the first sentence with key figures. Provide 1-2 bullet points highlighting critical context. Every metric must state its data source reference in brackets (e.g., [Source: SQL_Table_Q3_Revenue]). Do not include conversational greetings, fluff, or technical preamble."""

PROMPT_B = """You are Ascend IQ, an executive strategic analyst. 
Provide a narrative explanation of key business performance trends. Break down the background factors driving the numbers, compare current performance to historical benchmarks, and detail potential strategic implications. Cite source tables at the bottom of your report."""

JUDGE_SYSTEM_PROMPT = """You are an expert AI evaluator grading an executive AI assistant.
Evaluate the candidate response based on the original executive query.

Score the candidate response on a scale of 1-5 for each metric:
1. attribution_score (1-5): Are numbers clearly traceable to explicit data sources?
2. conciseness_score (1-5): Is the core answer delivered directly in sentence 1 without preamble/filler?
3. actionability_score (1-5): Is the output structured for immediate board/executive scanning?

Return STRICT JSON ONLY matching this format:
{
  "attribution_score": <1-5 integer>,
  "conciseness_score": <1-5 integer>,
  "actionability_score": <1-5 integer>,
  "reasoning": "<1-2 sentence explanation>"
}"""

# Sample dataset representing C-suite prompts
DATASET: List[Dict[str, Any]] = [
    {
        "id": 1,
        "role": "CFO",
        "query": "What was our Q3 ARR, and how does it compare to Q2?"
    },
    {
        "id": 2,
        "role": "CRO",
        "query": "What is our current Net Revenue Retention (NRR) across enterprise customer tiers?"
    },
    {
        "id": 3,
        "role": "CEO",
        "query": "Give me an update on CAC payback period for our SMB segment this quarter."
    }
]

def generate_llm_response(system_prompt: str, user_query: str, model: str) -> str:
    """Generate response from the target LLM generator."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content or ""

def judge_response(user_query: str, candidate_response: str, model: str) -> Dict[str, Any]:
    """Pass response to the judge model and return parsed JSON evaluation."""
    evaluation_prompt = f"""Executive Query: {user_query}

Candidate AI Response:
{candidate_response}"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": evaluation_prompt}
        ],
        temperature=0.0,
        response_format={"type": "json_object"}
    )
    
    raw_content = response.choices[0].message.content or "{}"
    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        cleaned = raw_content.replace("```json", "").replace("```", "").strip()
        return json.loads(cleaned)

def run_eval():
    results = {"Version_A": [], "Version_B": []}

    print("🚀 Starting Evaluation Harness Run...\n")

    # Loop pattern execution
    for item in DATASET:
        query_id = item["id"]
        role = item["role"]
        query = item["query"]
        
        print(f"--- Processing Query #{query_id} ({role}) ---")
        print(f"Query: \"{query}\"")

        # Step 1: Generate outputs with Generator Model (Prompt A & Prompt B)
        output_a = generate_llm_response(PROMPT_A, query, GENERATOR_MODEL)
        output_b = generate_llm_response(PROMPT_B, query, GENERATOR_MODEL)

        # Step 2: Pass generated outputs + original query to Judge Model
        eval_a = judge_response(query, output_a, JUDGE_MODEL)
        eval_b = judge_response(query, output_b, JUDGE_MODEL)

        results["Version_A"].append(eval_a)
        results["Version_B"].append(eval_b)

        print(f"  [Version A] Scores -> Attrib: {eval_a.get('attribution_score')} | Concise: {eval_a.get('conciseness_score')} | Actionable: {eval_a.get('actionability_score')}")
        print(f"  [Version B] Scores -> Attrib: {eval_b.get('attribution_score')} | Concise: {eval_b.get('conciseness_score')} | Actionable: {eval_b.get('actionability_score')}\n")

    # Step 3: Compute aggregate scores for Version A vs Version B
    summary = {}
    for version in ["Version_A", "Version_B"]:
        scores = results[version]
        avg_attrib = statistics.mean([s["attribution_score"] for s in scores])
        avg_concise = statistics.mean([s["conciseness_score"] for s in scores])
        avg_action = statistics.mean([s["actionability_score"] for s in scores])
        overall_avg = statistics.mean([avg_attrib, avg_concise, avg_action])

        summary[version] = {
            "Attribution": round(avg_attrib, 2),
            "Conciseness": round(avg_concise, 2),
            "Actionability": round(avg_action, 2),
            "Overall": round(overall_avg, 2)
        }

    # Print summary output table
    print("=" * 65)
    print("EVALUATION RESULTS SUMMARY")
    print("=" * 65)
    print(f"{'Metric':<25} | {'Version A (Concise)':<18} | {'Version B (Narrative)':<18}")
    print("-" * 65)
    print(f"{'Source Attribution':<25} | {summary['Version_A']['Attribution']:<18} | {summary['Version_B']['Attribution']:<18}")
    print(f"{'Executive Conciseness':<25} | {summary['Version_A']['Conciseness']:<18} | {summary['Version_B']['Conciseness']:<18}")
    print(f"{'Actionability':<25} | {summary['Version_A']['Actionability']:<18} | {summary['Version_B']['Actionability']:<18}")
    print("-" * 65)
    print(f"{'Overall Average':<25} | {summary['Version_A']['Overall']:<18} | {summary['Version_B']['Overall']:<18}")
    print("=" * 65)

    winner = "Version A (Concise)" if summary['Version_A']['Overall'] > summary['Version_B']['Overall'] else "Version B (Narrative)"
    print(f"\n🏆 Winner: {winner}\n")

if __name__ == "__main__":
    run_eval()

🚀 Starting Evaluation Harness Run...

--- Processing Query #1 (CFO) ---
Query: "What was our Q3 ARR, and how does it compare to Q2?"
  [Version A] Scores -> Attrib: 1 | Concise: 5 | Actionable: 5
  [Version B] Scores -> Attrib: 1 | Concise: 2 | Actionable: 3

--- Processing Query #2 (CRO) ---
Query: "What is our current Net Revenue Retention (NRR) across enterprise customer tiers?"
  [Version A] Scores -> Attrib: 1 | Concise: 2 | Actionable: 2
  [Version B] Scores -> Attrib: 1 | Concise: 1 | Actionable: 2

--- Processing Query #3 (CEO) ---
Query: "Give me an update on CAC payback period for our SMB segment this quarter."
  [Version A] Scores -> Attrib: 5 | Concise: 5 | Actionable: 5
  [Version B] Scores -> Attrib: 1 | Concise: 1 | Actionable: 2

EVALUATION RESULTS SUMMARY
Metric                    | Version A (Concise) | Version B (Narrative)
-----------------------------------------------------------------
Source Attribution        | 2.33               | 1                 
Executive C